In [3]:
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword = StopWordRemoverFactory().create_stop_word_remover()

dokumen2 = [
    "Sistem komputer dan jaringan",
    "Jaringan komputer berbasis kecerdasan buatan",
    "Kecerdasan buatan pada sistem temu kembali",
    "Sistem temu kembali informasi"
]

docs_clean = []
for d in dokumen2:
    teks = d.lower()
    teks = re.sub(r'[^a-z\s]', '', teks)
    teks = stopword.remove(teks)
    docs_clean.append(teks)

vocab = sorted(list(set(" ".join(docs_clean).split())))

tf = pd.DataFrame(np.zeros((len(docs_clean), len(vocab))), columns=vocab)
for i, doc in enumerate(docs_clean):
    for word in doc.split():
        tf.loc[i, word] += 1

df = (tf > 0).sum(axis=0)

N = len(docs_clean)
idf = np.log10(N / df)

tfidf_manual = tf * idf

vectorizer = TfidfVectorizer()
tfidf_sklearn = vectorizer.fit_transform(docs_clean)
df_sklearn = pd.DataFrame(tfidf_sklearn.toarray(), columns=vectorizer.get_feature_names_out())

print(" -- TERM FREQUENCY (TF) --")
print(tf)
print("\n -- DOCUMENT FREQUENCY (DF) & IDF --")
df_idf = pd.DataFrame({'DF': df, 'IDF': idf})
print(df_idf)
print("\n -- TF-IDF MANUAL --")
print(tfidf_manual)
print("\n -- TF-IDF SCIKIT-LEARN --")
print(df_sklearn)

 -- TERM FREQUENCY (TF) --
   berbasis  buatan  informasi  jaringan  kecerdasan  komputer  sistem  temu
0       0.0     0.0        0.0       1.0         0.0       1.0     1.0   0.0
1       1.0     1.0        0.0       1.0         1.0       1.0     0.0   0.0
2       0.0     1.0        0.0       0.0         1.0       0.0     1.0   1.0
3       0.0     0.0        1.0       0.0         0.0       0.0     1.0   1.0

 -- DOCUMENT FREQUENCY (DF) & IDF --
            DF       IDF
berbasis     1  0.602060
buatan       2  0.301030
informasi    1  0.602060
jaringan     2  0.301030
kecerdasan   2  0.301030
komputer     2  0.301030
sistem       3  0.124939
temu         2  0.301030

 -- TF-IDF MANUAL --
   berbasis   buatan  informasi  jaringan  kecerdasan  komputer    sistem  \
0   0.00000  0.00000    0.00000   0.30103     0.00000   0.30103  0.124939   
1   0.60206  0.30103    0.00000   0.30103     0.30103   0.30103  0.000000   
2   0.00000  0.30103    0.00000   0.00000     0.30103   0.00000  0.12493